In [12]:
using LowLevelFEM

In [13]:
line_mesh(n=3, order=4)

mat = Material("body")
P = Field([mat], type=:ScalarField, dim=1)

Problem("line_mesh", :ScalarField, 1, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 13, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :unknown, :rhs, false)

In [14]:
using SparseArrays
using LinearAlgebra


# =============================================================================
# Spectral-element helpers
# =============================================================================

"""
    _sem_legendre_pair(p, x)

Return `(Pp, Ppm1)`, where `Pp = P_p(x)` and `Ppm1 = P_{p-1}(x)`.
"""
function _sem_legendre_pair(p::Integer, x::Real)

    p ≥ 0 ||
        throw(ArgumentError("Polynomial order must be non-negative."))

    p == 0 && return (1.0, 0.0)

    P0 = 1.0
    P1 = Float64(x)

    p == 1 && return (P1, P0)

    for k in 2:p
        Pk =
            ((2k - 1) * x * P1 - (k - 1) * P0) / k

        P0, P1 = P1, Pk
    end

    return P1, P0
end


"""
    _sem_gll(p; tol=1e-14, maxiter=100)

Return the `p+1` Legendre-Gauss-Lobatto nodes and quadrature weights
on `[-1, 1]`.
"""
function _sem_gll(
    p::Integer;
    tol=1e-14,
    maxiter=100
)

    p ≥ 1 ||
        throw(ArgumentError("Polynomial order must be at least one."))

    ξ = zeros(Float64, p + 1)

    ξ[1] = -1.0
    ξ[end] = 1.0

    for k in 1:p-1

        x = -cos(pi * k / p)

        for _ in 1:maxiter

            Pp, Ppm1 =
                _sem_legendre_pair(p, x)

            dPp =
                p * (Ppm1 - x * Pp) /
                (1 - x^2)

            ddPp =
                (2x * dPp - p * (p + 1) * Pp) /
                (1 - x^2)

            Δx = dPp / ddPp

            x -= Δx

            abs(Δx) ≤ tol * max(1.0, abs(x)) &&
                break
        end

        ξ[k + 1] = x
    end

    sort!(ξ)

    w = similar(ξ)

    for i in eachindex(ξ)

        Pp, _ =
            _sem_legendre_pair(p, ξ[i])

        w[i] =
            2 / (p * (p + 1) * Pp^2)
    end

    return ξ, w
end


"""
    _sem_barycentric_weights(nodes)

Return barycentric interpolation weights for distinct interpolation nodes.
"""
function _sem_barycentric_weights(
    nodes::AbstractVector
)

    n = length(nodes)

    w = ones(Float64, n)

    for j in 1:n
        for k in 1:n

            j == k && continue

            w[j] /=
                nodes[j] - nodes[k]
        end
    end

    return w
end


"""
    _sem_lagrange_matrix(nodes, points; atol=1e-13)

Return the Lagrange interpolation matrix from `nodes` to `points`.
"""
function _sem_lagrange_matrix(
    nodes::AbstractVector,
    points::AbstractVector;
    atol=1e-13
)

    nodes = Float64.(nodes)
    points = Float64.(points)

    n = length(nodes)
    m = length(points)

    bw =
        _sem_barycentric_weights(nodes)

    N = zeros(Float64, m, n)

    for q in 1:m

        x = points[q]

        hit = findfirst(
            j -> isapprox(
                x,
                nodes[j];
                atol=atol,
                rtol=0.0
            ),
            1:n
        )

        if hit !== nothing

            N[q, hit] = 1.0

        else

            a =
                bw ./ (x .- nodes)

            N[q, :] .=
                a ./ sum(a)
        end
    end

    return N
end


"""
    _sem_insert_unique!(A, i, j, value; atol=1e-12)

Insert a sparse transformation entry. If the same global entry is generated
by two neighbouring elements, verify that the values are consistent.
"""
function _sem_insert_unique!(
    A::Dict{Tuple{Int,Int},Float64},
    i::Int,
    j::Int,
    value::Real;
    atol=1e-12
)

    abs(value) ≤ atol &&
        return nothing

    key = (i, j)

    v = Float64(value)

    if haskey(A, key)

        isapprox(
            A[key],
            v;
            atol=atol,
            rtol=atol
        ) ||
            error(
                "spectralTransformation: inconsistent transformation " *
                "entry ($i,$j): $(A[key]) vs. $v."
            )

    else

        A[key] = v
    end

    return nothing
end


"""
    _sem_sparse_from_dict(A, m, n)

Construct a sparse matrix from dictionary entries.
"""
function _sem_sparse_from_dict(
    A::Dict{Tuple{Int,Int},Float64},
    m::Int,
    n::Int
)

    I = Int[]
    J = Int[]
    V = Float64[]

    sizehint!(I, length(A))
    sizehint!(J, length(A))
    sizehint!(V, length(A))

    for ((i, j), value) in A

        push!(I, i)
        push!(J, j)
        push!(V, value)
    end

    return sparse(I, J, V, m, n)
end


# =============================================================================
# 1D spectral transformation
# =============================================================================

"""
    spectralTransformation(P::Problem; details=false)

Construct the transformation between the original Gmsh Lagrange nodal basis
and a Legendre-Gauss-Lobatto nodal basis for a one-dimensional finite-element
field.

The transformation satisfies

    u_fem = T * u_sem

and

    u_sem = R * u_fem

on the active finite-element space.

The implementation does not use Gmsh node tags as algebraic indices.
Instead, all Gmsh nodes are mapped to a compact internal numbering based on
their sorted node tags. This prepares the transformation for a future
model-wide compact DOF numbering.

For now, only one-dimensional Line elements with homogeneous polynomial order
are supported.

If `details=false`, return

    T, R

If `details=true`, return a named tuple containing `T`, `R`, node mappings,
spectral coordinates, element connectivity, and GLL data.
"""
function spectralTransformation(
    P::Problem;
    details::Bool=false
)

    gmsh.model.setCurrent(P.name)

    P.dim == 1 ||
        error(
            "spectralTransformation: only one-dimensional problems " *
            "are currently supported."
        )

    # ------------------------------------------------------------------
    # Global Gmsh nodes
    #
    # Gmsh tags are treated only as identifiers.
    # The algebraic node numbering is compact: 1, ..., number_of_nodes.
    # ------------------------------------------------------------------

    nodeTags0, coord0, _ =
        gmsh.model.mesh.getNodes(
            -1,
            -1,
            false,
            false
        )

    nodeCoord =
        Dict{UInt64,NTuple{3,Float64}}()

    sizehint!(
        nodeCoord,
        length(nodeTags0)
    )

    @inbounds for i in eachindex(nodeTags0)

        nodeCoord[nodeTags0[i]] = (
            coord0[3i - 2],
            coord0[3i - 1],
            coord0[3i]
        )
    end

    full_node_tags =
        sort(UInt64.(nodeTags0))

    length(full_node_tags) == P.non ||
        error(
            "spectralTransformation: Problem node count is inconsistent " *
            "with the current Gmsh model."
        )

    tag_to_full =
        Dict{UInt64,Int}()

    sizehint!(
        tag_to_full,
        length(full_node_tags)
    )

    for (i, tag) in enumerate(full_node_tags)
        tag_to_full[tag] = i
    end

    # This tells us whether the current LLFEM tag-based algebraic numbering
    # happens to coincide with the compact numbering used here.
    current_order_compatible =
        all(
            Int(tag) == i
            for (i, tag) in enumerate(full_node_tags)
        )

    # ------------------------------------------------------------------
    # Collect domain elements and cache element-type data
    # ------------------------------------------------------------------

    elements = NamedTuple[]

    seen_elements =
        Set{UInt64}()

    orders =
        Set{Int}()

    type_cache =
        Dict{Int,Any}()

    for mat in P.material

        dimTags =
            gmsh.model.getEntitiesForPhysicalName(
                mat.phName
            )

        for (edim, etag) in dimTags

            edim == 1 || continue

            elemTypes,
            elemTags,
            elemNodeTags =
                gmsh.model.mesh.getElements(
                    edim,
                    etag
                )

            for it in eachindex(elemTypes)

                et = elemTypes[it]

                name,
                dim,
                p,
                nFem,
                ξFem0,
                nPrimary =
                    gmsh.model.mesh.getElementProperties(
                        et
                    )

                occursin("Line", name) ||
                    error(
                        "spectralTransformation: unsupported 1D " *
                        "element type \"$name\"."
                    )

                dim == 1 ||
                    error(
                        "spectralTransformation: incompatible " *
                        "element dimension."
                    )

                push!(orders, p)

                if !haskey(type_cache, et)

                    ξFem =
                        Float64.(ξFem0)

                    length(ξFem) == nFem ||
                        error(
                            "spectralTransformation: invalid local " *
                            "coordinate data for element type \"$name\"."
                        )

                    ξGLL, wGLL =
                        _sem_gll(p)

                    nSem =
                        length(ξGLL)

                    nSem == nFem ||
                        error(
                            "spectralTransformation: expected the FEM and " *
                            "spectral spaces to have the same local dimension."
                        )

                    # --------------------------------------------------
                    # FEM values from spectral nodal values:
                    #
                    #     u_FEM = Te * u_GLL
                    # --------------------------------------------------

                    Te =
                        _sem_lagrange_matrix(
                            ξGLL,
                            ξFem
                        )

                    # --------------------------------------------------
                    # Spectral nodal values from FEM values:
                    #
                    #     u_GLL = Re * u_FEM
                    # --------------------------------------------------

                    Re =
                        _sem_lagrange_matrix(
                            ξFem,
                            ξGLL
                        )

                    type_cache[et] = (
                        name=name,
                        p=p,
                        nFem=nFem,
                        nSem=nSem,
                        nPrimary=nPrimary,
                        ξFem=ξFem,
                        ξGLL=ξGLL,
                        wGLL=wGLL,
                        Te=Te,
                        Re=Re
                    )
                end

                cache =
                    type_cache[et]

                tags =
                    elemTags[it]

                conn =
                    elemNodeTags[it]

                @inbounds for e in eachindex(tags)

                    elemTag =
                        tags[e]

                    elemTag in seen_elements &&
                        continue

                    push!(
                        seen_elements,
                        elemTag
                    )

                    o =
                        (e - 1) * cache.nFem

                    nodes =
                        UInt64.(
                            conn[
                                o + 1 : o + cache.nFem
                            ]
                        )

                    push!(
                        elements,
                        (
                            tag=elemTag,
                            et=et,
                            nodes=nodes,
                            primary=nodes[
                                1:cache.nPrimary
                            ]
                        )
                    )
                end
            end
        end
    end

    isempty(elements) &&
        error(
            "spectralTransformation: no 1D domain elements found."
        )

    length(orders) == 1 ||
        error(
            "spectralTransformation: homogeneous polynomial order " *
            "is currently required; found $(sort!(collect(orders)))."
        )

    p =
        first(orders)

    # ------------------------------------------------------------------
    # Characteristic length for physical-coordinate comparisons
    # ------------------------------------------------------------------

    used_nodes =
        Set{UInt64}()

    for elem in elements
        union!(
            used_nodes,
            elem.nodes
        )
    end

    xmin = Inf
    ymin = Inf
    zmin = Inf

    xmax = -Inf
    ymax = -Inf
    zmax = -Inf

    for tag in used_nodes

        x, y, z =
            nodeCoord[tag]

        xmin = min(xmin, x)
        ymin = min(ymin, y)
        zmin = min(zmin, z)

        xmax = max(xmax, x)
        ymax = max(ymax, y)
        zmax = max(zmax, z)
    end

    L =
        sqrt(
            (xmax - xmin)^2 +
            (ymax - ymin)^2 +
            (zmax - zmin)^2
        )

    L > 0 ||
        error(
            "spectralTransformation: degenerate mesh geometry."
        )

    tol =
        1e-10 * L

    tol2 =
        tol^2

    # ------------------------------------------------------------------
    # Global spectral node numbering
    # ------------------------------------------------------------------

    nElem =
        length(elements)

    spectral_conn =
        Vector{Vector{Int}}(
            undef,
            nElem
        )

    spectral_coord_local =
        Vector{Matrix{Float64}}(
            undef,
            nElem
        )

    spectral_coordinates =
        NTuple{3,Float64}[]

    vertex_to_spectral =
        Dict{UInt64,Int}()

    node_to_elements =
        Dict{UInt64,Vector{Int}}()

    nSpectral = 0

    for ie in 1:nElem

        elem =
            elements[ie]

        cache =
            type_cache[elem.et]

        # --------------------------------------------------------------
        # Original physical node coordinates
        # --------------------------------------------------------------

        Xfem =
            zeros(
                Float64,
                cache.nFem,
                3
            )

        for a in 1:cache.nFem

            x, y, z =
                nodeCoord[
                    elem.nodes[a]
                ]

            Xfem[a, 1] = x
            Xfem[a, 2] = y
            Xfem[a, 3] = z
        end

        # --------------------------------------------------------------
        # Physical coordinates of GLL nodes
        #
        # Geometry remains the original Gmsh isoparametric geometry.
        # --------------------------------------------------------------

        Xgll =
            cache.Re * Xfem

        spectral_coord_local[ie] =
            Xgll

        sconn =
            zeros(
                Int,
                cache.nSem
            )

        # --------------------------------------------------------------
        # Previously processed topological neighbours
        # --------------------------------------------------------------

        neighbours =
            Int[]

        for node in elem.primary
            append!(
                neighbours,
                get(
                    node_to_elements,
                    node,
                    Int[]
                )
            )
        end

        unique!(neighbours)

        # --------------------------------------------------------------
        # Spectral nodes
        # --------------------------------------------------------------

        for a in 1:cache.nSem

            ξ =
                cache.ξGLL[a]

            xa =
                Xgll[a, 1]

            ya =
                Xgll[a, 2]

            za =
                Xgll[a, 3]

            found = 0

            # ----------------------------------------------------------
            # Element vertices are identified exactly through the
            # original Gmsh vertex tag.
            # ----------------------------------------------------------

            if isapprox(
                abs(ξ),
                1.0;
                atol=1e-12,
                rtol=0.0
            )

                b =
                    argmin(
                        abs.(
                            cache.ξFem .- ξ
                        )
                    )

                abs(cache.ξFem[b] - ξ) < 1e-12 ||
                    error(
                        "spectralTransformation: could not identify " *
                        "a line endpoint."
                    )

                b <= cache.nPrimary ||
                    error(
                        "spectralTransformation: line endpoint is not " *
                        "stored as a primary Gmsh node."
                    )

                vertex_tag =
                    elem.nodes[b]

                found =
                    get(
                        vertex_to_spectral,
                        vertex_tag,
                        0
                    )

                if found == 0

                    nSpectral += 1

                    found =
                        nSpectral

                    vertex_to_spectral[
                        vertex_tag
                    ] = found

                    push!(
                        spectral_coordinates,
                        (xa, ya, za)
                    )
                end

            else

                # ------------------------------------------------------
                # Interior spectral nodes are searched only on
                # topologically connected processed elements.
                #
                # In 1D they are normally element-local, but this logic
                # is intentionally written in a form that can later be
                # generalized to shared edges and faces.
                # ------------------------------------------------------

                for je in neighbours

                    Xold =
                        spectral_coord_local[je]

                    cold =
                        spectral_conn[je]

                    for b in axes(Xold, 1)

                        dx =
                            xa - Xold[b, 1]

                        dy =
                            ya - Xold[b, 2]

                        dz =
                            za - Xold[b, 3]

                        if (
                            dx^2 +
                            dy^2 +
                            dz^2
                        ) <= tol2

                            found =
                                cold[b]

                            break
                        end
                    end

                    found != 0 &&
                        break
                end

                if found == 0

                    nSpectral += 1

                    found =
                        nSpectral

                    push!(
                        spectral_coordinates,
                        (xa, ya, za)
                    )
                end
            end

            sconn[a] =
                found
        end

        spectral_conn[ie] =
            sconn

        for node in elem.primary

            push!(
                get!(
                    node_to_elements,
                    node,
                    Int[]
                ),
                ie
            )
        end
    end

    # ------------------------------------------------------------------
    # Assemble scalar prolongation Tn
    #
    #     u_FEM = Tn * u_GLL
    # ------------------------------------------------------------------

    Tentries =
        Dict{
            Tuple{Int,Int},
            Float64
        }()

    for ie in 1:nElem

        elem =
            elements[ie]

        cache =
            type_cache[elem.et]

        sconn =
            spectral_conn[ie]

        for a in 1:cache.nFem

            full_node =
                tag_to_full[
                    elem.nodes[a]
                ]

            for b in 1:cache.nSem

                _sem_insert_unique!(
                    Tentries,
                    full_node,
                    sconn[b],
                    cache.Te[a, b]
                )
            end
        end
    end

    Tn =
        _sem_sparse_from_dict(
            Tentries,
            P.non,
            nSpectral
        )

    # ------------------------------------------------------------------
    # Assemble scalar restriction Rn
    #
    #     u_GLL = Rn * u_FEM
    #
    # A shared spectral node is taken from the first processed element
    # containing it.
    # ------------------------------------------------------------------

    Rentries =
        Dict{
            Tuple{Int,Int},
            Float64
        }()

    spectral_seen =
        falses(nSpectral)

    for ie in 1:nElem

        elem =
            elements[ie]

        cache =
            type_cache[elem.et]

        sconn =
            spectral_conn[ie]

        for a in 1:cache.nSem

            spectral_node =
                sconn[a]

            spectral_seen[
                spectral_node
            ] &&
                continue

            spectral_seen[
                spectral_node
            ] = true

            for b in 1:cache.nFem

                full_node =
                    tag_to_full[
                        elem.nodes[b]
                    ]

                _sem_insert_unique!(
                    Rentries,
                    spectral_node,
                    full_node,
                    cache.Re[a, b]
                )
            end
        end
    end

    Rn =
        _sem_sparse_from_dict(
            Rentries,
            nSpectral,
            P.non
        )

    # ------------------------------------------------------------------
    # Expand scalar transformation according to pdim
    # ------------------------------------------------------------------

    d =
        P.pdim

    Id =
        spdiagm(
            0 =>
            ones(Float64, d)
        )

    T =
        kron(Tn, Id)

    R =
        kron(Rn, Id)

    dropzeros!(T)
    dropzeros!(R)

    # ------------------------------------------------------------------
    # Global spectral coordinates
    # ------------------------------------------------------------------

    Xspectral =
        zeros(
            Float64,
            nSpectral,
            3
        )

    for i in 1:nSpectral
        Xspectral[i, :] .=
            spectral_coordinates[i]
    end

    if !details
        return T, R
    end

    # For homogeneous order all element types have identical GLL nodes,
    # but obtain them without assuming a particular element type.
    first_cache =
        type_cache[
            elements[1].et
        ]

    return (
        T=T,
        R=R,
        order=p,

        full_node_tags=
            full_node_tags,

        tag_to_full=
            tag_to_full,

        current_order_compatible=
            current_order_compatible,

        spectral_coordinates=
            Xspectral,

        spectral_connectivity=
            spectral_conn,

        element_tags=
            [elem.tag for elem in elements],

        ξ_gll=
            first_cache.ξGLL,

        w_gll=
            first_cache.wGLL
    )
end

spectralTransformation (generic function with 1 method)

In [15]:
sem =
    spectralTransformation(
        P;
        details=true
    )

T = sem.T
R = sem.R

@show sem.order
@show size(T)
@show size(R)
@show nnz(T)
@show nnz(R)

@show sem.current_order_compatible

Isem =
    spdiagm(
        0 =>
        ones(Float64, size(R, 1))
    )

@show norm(
    R * T - Isem,
    Inf
)

display(
    sem.spectral_coordinates
)

sem.order = 4
size(T) = (13, 13)
size(R) = (13, 13)
nnz(T) = 37
nnz(R) = 37
sem.current_order_compatible = true
norm(R * T - Isem, Inf) = 4.440892098500626e-16


13×3 Matrix{Float64}:
 0.0        0.0  0.0
 0.0575577  0.0  0.0
 0.166667   0.0  0.0
 0.275776   0.0  0.0
 0.333333   0.0  0.0
 0.390891   0.0  0.0
 0.5        0.0  0.0
 0.609109   0.0  0.0
 0.666667   0.0  0.0
 0.724224   0.0  0.0
 0.833333   0.0  0.0
 0.942442   0.0  0.0
 1.0        0.0  0.0